# SmolDLM Phase 4: Modern Block Diffusion Training

Train a modern block diffusion LLM on multi-source 100B dataset using Kaggle 2xT4 GPUs.

- **Model**: ~125M params (20L/768d/12h/3kv/1536MLP, tied embeddings), block_size=32
- **Data**: FinePDFs 50B + DCLM 30B + FineWeb-Edu 20B (pre-shuffled, streaming)
- **Key innovations**: FlexAttention staircase mask, GQA 4:1, Liger fused kernels,
  AMP bf16, gradient checkpointing, complementary masking,
  WSD scheduler, Muon optimizer, MLP dropout 0.1, torch.compile
- **Hardware**: 2xT4 (CC 7.5, 16GB each) with DDP multi-GPU training
- **Default config**: seq_len=2048, batch=64, grad_accum=1 (0.5M tok/step on A100)
- **Kaggle override**: seq_len=1024, batch=16, grad_accum=2 (T4 16GB VRAM constraint)
- **References**: ZHZisZZ/dllm (BD3-LMs), JinjieNi/MegaDLMs (Megatron-based DLMs)

In [ ]:
# Install deps — liger-kernel requires CC 7.0+ (T4 = CC 7.5)
!pip install -q torch datasets tokenizers liger-kernel

# muon: KellerJordan's optimizer (NOT the bioinformatics 'muon' package on PyPI)
# Retry up to 3 times for transient GitHub errors
!pip uninstall -y muon 2>/dev/null
import subprocess, time
for attempt in range(3):
    r = subprocess.run(
        ["pip", "install", "-q", "git+https://github.com/KellerJordan/Muon"],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f"Muon installed (attempt {attempt + 1})")
        break
    print(f"Muon install attempt {attempt + 1} failed")
    if attempt < 2:
        time.sleep(30)
else:
    print("WARNING: Muon install failed — will use AdamW fallback (--no-muon)")

In [ ]:
import subprocess, time, os
# Retry git clone up to 5 times (handles transient GitHub 500 errors)
for attempt in range(5):
    result = subprocess.run(
        ["git", "clone", "https://github.com/hahuyhoang411/SmolDLM.git"],
        capture_output=True, text=True
    )
    if result.returncode == 0 or os.path.isdir("SmolDLM"):
        print(f"Clone succeeded (attempt {attempt + 1})")
        break
    print(f"Clone attempt {attempt + 1} failed: {result.stderr.strip()}")
    if attempt < 4:
        time.sleep(30 * (attempt + 1))  # 30s, 60s, 90s, 120s backoff
else:
    raise RuntimeError("git clone failed after 5 attempts — GitHub may be down")
%cd SmolDLM

In [ ]:
import torch
n_gpus = torch.cuda.device_count()
print(f"GPUs: {n_gpus}")
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    cc = torch.cuda.get_device_capability(i)
    print(f"  [{i}] {name} — {vram:.1f} GB — CC {cc[0]}.{cc[1]}")

print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

cc = torch.cuda.get_device_capability(0)
if cc >= (7, 0):
    print(f"\nCC {cc[0]}.{cc[1]} — Liger Kernel + FlexAttention + torch.compile compatible")
else:
    print(f"\nCC {cc[0]}.{cc[1]} — will use fallbacks (no Liger/FlexAttention/compile)")

In [ ]:
# Train with DDP on 2xT4: torchrun handles multi-GPU setup
# --no-compile --no-flex: FlexAttention internally uses torch.compile which hits
# T4 Triton shared memory limits (65536 bytes). SDPA fallback is still fast.
# --no-grad-ckpt: disable gradient checkpointing — trades VRAM for ~25% faster training.
# Liger fused kernels (SwiGLU, CrossEntropy, RMSNorm) provide the main speedup.
# CART is off by default (uniform 1/t ELBO weights). Add --cart to enable.
# Expected step-0 loss: ~10-11 (ln(32768)). If >>20, check loss computation.
# Architecture defaults: 20L/768d/12h/3kv/1536MLP (~125M)
# Code defaults: seq_len=2048, batch=64, grad_accum=1 (for A100 40GB)
# Kaggle T4 override: seq_len=1024, batch=16, grad_accum=2 (fits T4 16GB)
import torch
n_gpus = torch.cuda.device_count()

if n_gpus > 1:
    # DDP on T4: override defaults for 16GB VRAM
    # effective batch = 16 * 2 * 2 * 2 * 1024 = 131,072 tokens/step
    !torchrun --nproc_per_node={n_gpus} 04_modern_dllm/modern_dllm.py \
        --train --seq-len 1024 --batch-size 16 --grad-accum-steps 2 \
        --no-compile --no-flex --no-grad-ckpt
else:
    # Single GPU T4: higher grad_accum to compensate
    !python 04_modern_dllm/modern_dllm.py --train \
        --seq-len 1024 --batch-size 16 --grad-accum-steps 4 --no-grad-ckpt

In [ ]:
# Generate sample text
!python 04_modern_dllm/modern_dllm.py \
    --prompt "The meaning of life is" --max-tokens 128 --denoise-steps 10

In [ ]:
import shutil, os
os.makedirs('/kaggle/working/weights', exist_ok=True)
src = '04_modern_dllm/weights/modern_dllm_b32.pt'
if os.path.exists(src):
    shutil.copy(src, '/kaggle/working/weights/')
    size_mb = os.path.getsize(f'/kaggle/working/weights/{os.path.basename(src)}') / 1e6
    print(f"Saved weights to /kaggle/working/weights/ ({size_mb:.1f} MB)")
else:
    print(f"Weights not found at {src}")